In [1]:
import sys
sys.path.append("../src")

from pathlib import Path
import joblib
import numpy as np
import pandas as pd

import lightgbm as lgb
from sklearn.model_selection import train_test_split

from varclushi import VarClusHi
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROCESSED = Path("../data/processed")
ARTEFACTS = Path("../outputs/models")

In [2]:
df_train = pd.read_parquet(PROCESSED / "woe_train.parquet")
df_test  = pd.read_parquet(PROCESSED / "woe_test.parquet")

binning_process = joblib.load(ARTEFACTS / "binning_process.pkl")
binning_summary = pd.read_csv(PROCESSED / "binning_summary.csv")
iv_lookup = binning_summary.set_index("name")["iv"].to_dict()

feature_cols = [c for c in df_train.columns if c not in ("SK_ID_CURR", "TARGET")]
X_train = df_train[feature_cols]
y_train = df_train["TARGET"]

print(f"Train shape: {df_train.shape}")
print(f"Test shape:  {df_test.shape}")
print(f"Features for selection: {len(feature_cols)}")

Train shape: (307511, 119)
Test shape:  (48744, 118)
Features for selection: 117


In [3]:
# EMERGENCYSTATE_MODE and HOUSETYPE_MODE showed n_bins=1 yet non-zero IV.
# Confirm whether this is a counting convention or a genuine problem.
print(binning_process.get_binned_variable("EMERGENCYSTATE_MODE").binning_table.build())

                Bin   Count  Count (%)  Non-event  Event  Event rate     WoE  \
0       (-inf, inf)  161756     0.5260     150429  11327      0.0700  0.1538   
1           Special       0     0.0000          0      0      0.0000  0.0000   
2           Missing  145755     0.4740     132257  13498      0.0926 -0.1503   
Totals               307511     1.0000     282686  24825      0.0807           

           IV     JS  
0      0.0117 0.0015  
1      0.0000 0.0000  
2      0.0114 0.0014  
Totals 0.0231 0.0029  


In [4]:
# DEBT_TO_INCOME (IV 0.007) screened out despite being an EDA-mandated ratio.
# Check whether the monotonic constraint forced a poor split.
print(binning_process.get_binned_variable("DEBT_TO_INCOME").binning_table.build())

                 Bin   Count  Count (%)  Non-event  Event  Event rate     WoE  \
0       (-inf, 0.06)   16715     0.0544      15601   1114      0.0666  0.2069   
1       [0.06, 0.10)   45934     0.1494      42560   3374      0.0735  0.1023   
2       [0.10, 0.16)   82746     0.2691      76330   6416      0.0775  0.0438   
3       [0.16, 0.20)   52319     0.1701      47902   4417      0.0844 -0.0488   
4        [0.20, inf)  109797     0.3571     100293   9504      0.0866 -0.0761   
5            Special       0     0.0000          0      0      0.0000  0.0000   
6            Missing       0     0.0000          0      0      0.0000  0.0000   
Totals                307511     1.0000     282686  24825      0.0807           

           IV     JS  
0      0.0021 0.0003  
1      0.0015 0.0002  
2      0.0005 0.0001  
3      0.0004 0.0001  
4      0.0021 0.0003  
5      0.0000 0.0000  
6      0.0000 0.0000  
Totals 0.0067 0.0008  


In [5]:
vc = VarClusHi(X_train, maxeigval2=1.0, maxclus=None)
vc.varclus()
print(vc.info)

   Cluster N_Vars  Eigval1  Eigval2  VarProp
0        0     13  11.0882   0.9230   0.8529
1        1     14  10.6509   0.9913   0.7608
2        2      8   5.7013   0.9268   0.7127
3        3      9   5.9756   0.9082   0.6640
4        4      4   3.4246   0.2961   0.8561
5        5      7   4.1792   0.9062   0.5970
6        6      7   3.9131   0.9750   0.5590
7        7      3   2.0182   0.7335   0.6727
8        8      7   5.7480   0.7513   0.8211
9        9      3   2.5670   0.3882   0.8557
10      10      2   1.9221   0.0779   0.9611
11      11      5   2.4656   0.9868   0.4931
12      12      6   3.9171   0.9974   0.6528
13      13      3   1.9490   0.9797   0.6497
14      14      2   1.8016   0.1984   0.9008
15      15      2   1.6356   0.3644   0.8178
16      16      2   1.9685   0.0315   0.9842
17      17      2   1.4404   0.5596   0.7202
18      18      3   2.9425   0.0527   0.9808
19      19      2   1.3852   0.6148   0.6926
20      20      2   1.3049   0.6951   0.6524
21      21

In [6]:
rsquare = vc.rsquare.copy()
rsquare["IV"] = rsquare["Variable"].map(iv_lookup)

print(f"Variables: {len(rsquare)}")
print(f"Clusters:  {rsquare['Cluster'].nunique()}")
print(rsquare.to_string(index=False))


Variables: 117
Clusters:  25
 Cluster                        Variable  RS_Own  RS_NC  RS_Ratio     IV
       0                  APARTMENTS_AVG  0.8685 0.6430    0.3683 0.0326
       0                   ELEVATORS_AVG  0.8075 0.4265    0.3356 0.0332
       0                   FLOORSMAX_AVG  0.8626 0.5002    0.2749 0.0384
       0                  LIVINGAREA_AVG  0.8866 0.6252    0.3027 0.0343
       0                 APARTMENTS_MODE  0.8388 0.6477    0.4574 0.0318
       0                  ELEVATORS_MODE  0.8050 0.4332    0.3441 0.0320
       0                  FLOORSMAX_MODE  0.8639 0.5102    0.2779 0.0376
       0                 LIVINGAREA_MODE  0.8618 0.6266    0.3702 0.0332
       0                 APARTMENTS_MEDI  0.8630 0.6492    0.3906 0.0323
       0                  ELEVATORS_MEDI  0.8093 0.4251    0.3318 0.0329
       0                  FLOORSMAX_MEDI  0.8676 0.5057    0.2679 0.0381
       0                 LIVINGAREA_MEDI  0.8878 0.6298    0.3032 0.0341
       0              

In [7]:
# Selection rule: highest IV within cluster, RS_Ratio as tiebreaker.
# Guard: variables with RS_Ratio >= 1 are better explained by another
# cluster and cannot represent their own — retained separately as orphans.

BAD_FIT_THRESHOLD = 1.0

valid = rsquare[rsquare["RS_Ratio"] < BAD_FIT_THRESHOLD].copy()

selected = (
    valid.sort_values(["Cluster", "IV", "RS_Ratio"], ascending=[True, False, True])
         .groupby("Cluster")
         .first()
         .reset_index()
)

orphans = rsquare[rsquare["RS_Ratio"] >= BAD_FIT_THRESHOLD].copy()

vc_selected_features = selected["Variable"].tolist() + orphans["Variable"].tolist()

print(f"Cluster representatives: {len(selected)}")
print(f"Orphans retained:        {len(orphans)}")
print(f"Total after VARCLUS:     {len(vc_selected_features)}\n")

print("--- Representatives ---")
print(selected[["Cluster", "Variable", "RS_Ratio", "IV"]].to_string(index=False))
print("\n--- Orphans ---")
print(orphans[["Cluster", "Variable", "RS_Own", "RS_Ratio", "IV"]].to_string(index=False))

Cluster representatives: 25
Orphans retained:        2
Total after VARCLUS:     27

--- Representatives ---
 Cluster                        Variable  RS_Ratio     IV
       0                   FLOORSMAX_AVG    0.2749 0.0384
       1         CC_FLAG_EVER_OVER_LIMIT    0.3988 0.0385
       2                   EXT_1_2_3_MIN    0.5565 0.4643
       3         INST_MEAN_PAYMENT_RATIO    0.4446 0.0541
       4                    EXT_1_3_MEAN    0.0652 0.4144
       5                    REFUSAL_RATE    0.2382 0.0700
       6                     YEARS_BIRTH    0.5380 0.0857
       7         DAYS_SINCE_FIRST_CREDIT    0.5086 0.0782
       8                   ENTRANCES_AVG    0.2992 0.0277
       9     REGION_RATING_CLIENT_W_CITY    0.0939 0.0512
      10                 AMT_GOODS_PRICE    0.0402 0.0531
      11            DEBT_TO_CREDIT_RATIO    0.3675 0.1028
      12             CC_MEAN_UTILISATION    0.2994 0.0615
      13                  YEARS_EMPLOYED    0.0448 0.1131
      14              

In [8]:
def stepwise_vif(X, threshold=5.0, verbose=True):
    """
    Iteratively drop the highest-VIF feature until all remain below threshold.
    threshold: 5 = strict (regulatory scorecard standard), 10 = lenient
    """
    features = list(X.columns)
    dropped = []

    while True:
        Xc = add_constant(X[features], has_constant="add")
        vifs = pd.Series(
            [variance_inflation_factor(Xc.values, i) for i in range(1, Xc.shape[1])],
            index=features,
        )
        worst_vif = vifs.max()
        if worst_vif < threshold:
            break
        worst_feature = vifs.idxmax()
        features.remove(worst_feature)
        dropped.append((worst_feature, round(worst_vif, 2)))
        if verbose:
            print(f"Dropped {worst_feature:<35} VIF = {worst_vif:.2f}")

    return features, dropped, vifs


X_vc = X_train[vc_selected_features]
vif_features, vif_dropped, final_vifs = stepwise_vif(X_vc, threshold=5.0)

print(f"\nBefore VIF: {len(vc_selected_features)}")
print(f"After VIF:  {len(vif_features)}")
print(f"Dropped:    {len(vif_dropped)}\n")
print("--- Final VIFs ---")
print(final_vifs.sort_values(ascending=False).round(2).to_string())


Before VIF: 27
After VIF:  27
Dropped:    0

--- Final VIFs ---
YEARS_BEGINEXPLUATATION_MEDI      2.2700
FLOORSMAX_AVG                     2.2400
NONLIVINGAREA_AVG                 2.1100
EXT_1_2_3_MIN                     2.1000
EXT_1_3_MEAN                      2.0800
ENTRANCES_AVG                     1.9400
CC_FLAG_EVER_OVER_LIMIT           1.6700
CC_MEAN_UTILISATION               1.6100
EXT_SOURCE_1                      1.4200
EXT_2_3_STD                       1.3600
YEARS_BIRTH                       1.2600
INST_COUNT_LATE_LAST_12M          1.2300
DAYS_SINCE_FIRST_CREDIT           1.2200
DEBT_TO_CREDIT_RATIO              1.2200
AMT_GOODS_PRICE                   1.2100
INST_MEAN_PAYMENT_RATIO           1.1600
REG_CITY_NOT_WORK_CITY            1.1500
YEARS_EMPLOYED                    1.1500
ANNUITY_TO_CREDIT                 1.1400
REGION_RATING_CLIENT_W_CITY       1.1300
CREDIT_TO_GOODS                   1.1200
MEAN_AMT_CREDIT_SUM               1.1100
INST_MEAN_AMT_INSTALMENT         

In [9]:
final_features = vif_features

scorecard_train = df_train[["SK_ID_CURR", "TARGET"] + final_features]
scorecard_test  = df_test[["SK_ID_CURR"] + final_features]

scorecard_train.to_parquet(PROCESSED / "scorecard_train.parquet", index=False)
scorecard_test.to_parquet(PROCESSED / "scorecard_test.parquet", index=False)

joblib.dump(final_features, ARTEFACTS / "scorecard_features.pkl")

print(f"Scorecard train: {scorecard_train.shape}")
print(f"Scorecard test:  {scorecard_test.shape}")
print(f"\nFinal feature set ({len(final_features)}):")
for f in sorted(final_features):
    print(f"  {f}")

Scorecard train: (307511, 29)
Scorecard test:  (48744, 28)

Final feature set (27):
  AMT_GOODS_PRICE
  ANNUITY_TO_CREDIT
  CC_FLAG_EVER_OVER_LIMIT
  CC_MEAN_UTILISATION
  CREDIT_TO_GOODS
  DAYS_SINCE_FIRST_CREDIT
  DEBT_TO_CREDIT_RATIO
  ENTRANCES_AVG
  EXT_1_2_3_MIN
  EXT_1_3_MEAN
  EXT_2_3_STD
  EXT_SOURCE_1
  FLOORSMAX_AVG
  INST_COUNT_LATE_LAST_12M
  INST_MEAN_AMT_INSTALMENT
  INST_MEAN_PAYMENT_RATIO
  MEAN_AMT_CREDIT_SUM
  MEAN_CREDIT_TO_APP_RATIO
  NONLIVINGAREA_AVG
  OCCUPATION_TYPE
  POSITIVE_DAYS_LAST_PHONE_CHANGE
  REFUSAL_RATE
  REGION_RATING_CLIENT_W_CITY
  REG_CITY_NOT_WORK_CITY
  YEARS_BEGINEXPLUATATION_MEDI
  YEARS_BIRTH
  YEARS_EMPLOYED


In [14]:
# --- Load challenger track (raw features, not WoE) ---
challenger_train = pd.read_parquet(PROCESSED / "challenger_train.parquet")

X_ch = challenger_train.drop(columns=["SK_ID_CURR", "TARGET"])
y_ch = challenger_train["TARGET"]

# Categorical columns need encoding for LightGBM
cat_cols_ch = X_ch.select_dtypes(include=["object", "bool", "category"]).columns.tolist()
for c in cat_cols_ch:
    X_ch[c] = X_ch[c].astype("category")

# --- Quick model for importance ranking only (not tuned) ---
X_tr, X_val, y_tr, y_val = train_test_split(
    X_ch, y_ch, test_size=0.2, stratify=y_ch, random_state=42
)

model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=31,
    min_child_samples=100,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    is_unbalance=True,
    metric="auc",              # ← track ONLY auc, not logloss
    first_metric_only=True,    # ← early stopping uses the first metric
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    eval_metric="auc",
    callbacks=[
        lgb.early_stopping(100, first_metric_only=True, verbose=True),
        lgb.log_evaluation(100),
    ],
)

print(f"\nBest iteration: {model.best_iteration_}")
print(f"Validation AUC: {model.best_score_['valid_0']['auc']:.4f}")

importance = pd.DataFrame({
    "feature": X_ch.columns,
    "gain": model.booster_.feature_importance(importance_type="gain"),
}).sort_values("gain", ascending=False).reset_index(drop=True)

importance["rank_gain"] = range(1, len(importance) + 1)
importance["IV"] = importance["feature"].map(iv_lookup)
importance["in_scorecard"] = importance["feature"].isin(final_features)

print(f"Features with non-zero gain: {(importance['gain'] > 0).sum()} of {len(importance)}")
print("\n--- Top 30 by gain ---")
print(importance.head(30).to_string(index=False))

Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.778186
[200]	valid_0's auc: 0.784453
[300]	valid_0's auc: 0.785826
[400]	valid_0's auc: 0.78611
[500]	valid_0's auc: 0.786507
[600]	valid_0's auc: 0.786555
Early stopping, best iteration is:
[530]	valid_0's auc: 0.786902
Evaluated only: auc

Best iteration: 530
Validation AUC: 0.7869
Features with non-zero gain: 215 of 247

--- Top 30 by gain ---
                  feature         gain  rank_gain     IV  in_scorecard
           EXT_1_2_3_MEAN 247,315.8634          1 0.6086         False
        ORGANIZATION_TYPE 220,926.8885          2 0.0712         False
             EXT_2_3_MEAN 176,545.8971          3 0.5385         False
        ANNUITY_TO_CREDIT  53,565.7504          4 0.0290          True
             EXT_1_3_MEAN  50,443.8951          5 0.4144          True
          OCCUPATION_TYPE  41,346.3586          6 0.0796          True
 MEAN_CREDIT_TO_APP_RATIO  34,875.4726          7 0.0715          Tru

In [15]:
top30_lgb = set(importance.head(30)["feature"])
scorecard_set = set(final_features)

print(f"Scorecard features in LightGBM top 30: {len(scorecard_set & top30_lgb)} of {len(scorecard_set)}\n")

print("--- Scorecard features NOT in LightGBM top 30 ---")
missed = importance[importance["in_scorecard"] & (importance["rank_gain"] > 30)]
print(missed[["feature", "rank_gain", "gain", "IV"]].to_string(index=False))

print("\n--- LightGBM top 30 NOT in scorecard ---")
extra = importance.head(30)[~importance.head(30)["in_scorecard"]]
print(extra[["feature", "rank_gain", "gain", "IV"]].to_string(index=False))

Scorecard features in LightGBM top 30: 14 of 27

--- Scorecard features NOT in LightGBM top 30 ---
                        feature  rank_gain        gain     IV
            MEAN_AMT_CREDIT_SUM         32 15,850.2837 0.0257
POSITIVE_DAYS_LAST_PHONE_CHANGE         35 14,748.3837 0.0474
            CC_MEAN_UTILISATION         44 12,070.4950 0.0615
        DAYS_SINCE_FIRST_CREDIT         45 11,894.7380 0.0782
                    EXT_2_3_STD         47 11,336.5642 0.0388
       INST_COUNT_LATE_LAST_12M         72  6,435.0288 0.0618
    REGION_RATING_CLIENT_W_CITY         79  4,866.3368 0.0512
   YEARS_BEGINEXPLUATATION_MEDI         97  3,268.3439 0.0293
              NONLIVINGAREA_AVG        102  3,105.2439 0.0231
                  ENTRANCES_AVG        124  2,022.4747 0.0277
                  FLOORSMAX_AVG        138  1,446.9177 0.0384
         REG_CITY_NOT_WORK_CITY        194    184.9876 0.0322
        CC_FLAG_EVER_OVER_LIMIT        217      0.0000 0.0385

--- LightGBM top 30 NOT in score

In [16]:
# --- PCA: documented exclusion ---
#
# SCORECARD TRACK — PCA is EXCLUDED.
# Principal components are linear combinations of WoE-transformed
# features with no business meaning. A scorecard must state
# "applicant scored -15 points for high credit utilisation", not
# "-15 points for component 3". PCA is incompatible with the
# interpretability and regulatory-explainability requirements that
# justify building a scorecard at all.
#
# CHALLENGER TRACK — PCA is UNNECESSARY.
# Gradient boosting is invariant to monotonic feature transformations
# and handles correlated features natively through split selection.
# PCA would add preprocessing complexity, destroy the native
# categorical handling LightGBM provides, and prevent SHAP
# explanations at the original-feature level required in Phase 8.
# Validation AUC of 0.787 on 247 raw features confirms no
# dimensionality problem requiring reduction.
#
# DECISION: PCA not applied to either track.

print("PCA excluded from both tracks — see rationale above.")

PCA excluded from both tracks — see rationale above.
